# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 human mast cell extracellular vesicle protein dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant metadata URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets (`@id`s), each field (`@id`), and their column structure. We will print available record sets and preview one record for each.

In [ ]:
# List all available record sets and their fields with @ids
from mlcroissant._dataset_metadata import Metadata  # for type hinting (optional)

record_sets = dataset.metadata.record_sets

print("Record sets found in dataset:\n")
for rs in record_sets:
    print(f"- Record set @id: {rs.id}")
    print(f"  name: {rs.name}")
    print(f"  description: {rs.description}")
    print(f"  Fields (by @id):")
    for field in rs.fields:
        print(f"    - {field.id}")
    print("")
# Print a sample record for each record set
for rs in record_sets:
    print(f"Sample record for record set: {rs.id}")
    records_iter = dataset.records(record_set=rs.id)
    try:
        print(next(records_iter))
    except StopIteration:
        print("  No records found.")
    print("")

## 3. Data Extraction
Load data from each record set into a DataFrame for further analysis, referencing record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets into dataframes (by @id)
dataframes = dict()
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Record set: {rsid}, loaded {len(df)} rows, columns: {df.columns.tolist()}")
# For demonstration, pick the first record set for deeper analysis
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"\nFirst 5 rows for record set {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records by a numeric field, normalizing values, and grouping data, referencing field `@id`s.

In [ ]:
# Pick a numeric field (by @id) to filter and normalize; adjust as per the real field names discovered above
main_df = dataframes[main_record_set_id]

# Attempt to auto-select the first numeric column (float/int, heuristically)
numeric_field_id = None
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    # fallback to common protein quant columns
    for probe in ["cr:abundance_total", "cr:molecular_weight", "cr:peptide_count", "cr:coverage_percent"]:
        if probe in main_df.columns:
            numeric_field_id = probe
            break

if numeric_field_id is not None:
    print(f"Using numeric field '{numeric_field_id}' for EDA.")
    threshold = (main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype != 'O' else 10)
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try grouping by a categorical field, e.g., protein description, accession, or group/sample
    group_field_id = None
    # prioritize possible group fields
    for key in ["cr:protein_group", "cr:description", "cr:accession", "cr:sample_id"]:
        if key in main_df.columns:
            group_field_id = key
            break

    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (showing head):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize numeric field distributions or relationships. This example plots a histogram and a boxplot for the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in main_df.columns:
    plt.figure(figsize=(10,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    plt.figure(figsize=(6,4))
    sns.boxplot(x=main_df[numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.show()
    
    if group_field_id is not None and group_field_id in main_df.columns:
        plt.figure(figsize=(12,4))
        sns.violinplot(data=main_df, x=group_field_id, y=numeric_field_id, inner=None)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field for visualization.")

## 6. Conclusion
In this notebook we loaded the FAIR^2 dataset Croissant package, explored available record sets and fields referencing their `@id`s, extracted record sets into pandas DataFrames, filtered and normalized a numeric field, grouped data, and created basic visualizations. Please consult the fields and data dictionaries associated with each record set (see the Croissant schema/metadata for details on each field's meaning).